<div style="border:1px solid #d9e1ea;border-left:6px solid #2d8a57;border-radius:14px;padding:18px 20px;background:#fff;"><h1 style="margin:0 0 6px;color:#16213b;">LlamaIndex — Complete Learning Notebook</h1><p style="margin:0;color:#63738a;">Documents, Nodes, indexes, storage, retrieval, query engines and response synthesis — mapped to app/rag/service.py.</p><p style="margin:10px 0 0;"><code>notebooks/llamaindex/llamaindex_project_learning.ipynb</code></p></div>

![RAG Pipeline](assets/01_rag_pipeline.svg)

## 1. What LlamaIndex Is

LlamaIndex is a framework for connecting LLMs to your own data.

LlamaIndex = a data and retrieval framework for building RAG applications with LLMs.

It is mainly used for:

```
document ingestion
chunking and indexing
embeddings
retrieval
RAG
querying private knowledge
```

Knowledge definitions → LlamaIndex RAG. Actual reserved-user counts → Controlled SQL.

In [5]:
from pathlib import Path

current = Path.cwd()
project_root = current.parents[1]

print("Current:", current)
print("Two levels up:", project_root)

knowledge_dir = Path.cwd().parent.parent / "knowledge"

print("Knowledge directory:", knowledge_dir)
print("Exists:", knowledge_dir.exists())

Current: /Users/B1/ghome/github/online/reservation-analytics-ai-agent/notebooks/llamaindex
Two levels up: /Users/B1/ghome/github/online/reservation-analytics-ai-agent
Knowledge directory: /Users/B1/ghome/github/online/reservation-analytics-ai-agent/knowledge
Exists: True


## 2. Core Object Model

![Objects](assets/02_objects.svg)

| Object | Meaning |
|---|---|
| `Document` | Loaded source |
| `Node` | Smaller retrieval unit |
| `VectorStoreIndex` | Index abstraction |
| `Retriever` | Finds relevant nodes |
| `QueryEngine` | Retrieval + answer synthesis |
| `StorageContext` | Connects storage/vector components |

## 3. Loading Documents

In [7]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_dir=str(knowledge_dir),
    required_exts=[".md"],
).load_data()

## 4. Nodes / Chunking

Chunk size controls retrieval granularity.  
Too large adds noise; too small fragments context.   
The current project keeps defaults for simplicity.

**QA：**  
For this prototype, I kept the default chunking configuration for simplicity. In production, I would tune chunk size and overlap based on retrieval quality.

## 5. Embeddings

In [8]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(model="text-embedding-3-small")

## 6. VectorStoreIndex + StorageContext

In [9]:
vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(1536))  # 1536 = embedding vector dimension
storage = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage)

NameError: name 'FaissVectorStore' is not defined

## 7. Retriever vs Query Engine

Retriever finds nodes. Query Engine typically retrieves and then asks an LLM to synthesize the answer.

## 8. Top K

## 9. Current Project Mapping

![Project](assets/03_project.svg)

In [10]:
self._engine = index.as_query_engine(similarity_top_k=3)
return str(self._engine.query(question))

NameError: name 'index' is not defined

## 10. Production Improvements

Explicit chunking, metadata filters, persisted index, hybrid retrieval, reranking, evaluation dataset, citations and tracing.

## Q&A — Fast Review

<details open><summary><b>Q1. Document vs Node?</b></summary>

**Answer:** Document is the loaded source; Node is a smaller retrieval unit.

</details>

<details open><summary><b>Q2. Retriever vs Query Engine?</b></summary>

**Answer:** Retriever finds nodes; QueryEngine usually adds response synthesis.

</details>

<details open><summary><b>Q3. Why LlamaIndex here?</b></summary>

**Answer:** It keeps the knowledge layer concise while numbers stay in SQL.

</details>

<details open><summary><b>Q4. What does StorageContext do?</b></summary>

**Answer:** It connects index storage components such as FAISS.

</details>

<details open><summary><b>Q5. How to debug bad RAG?</b></summary>

**Answer:** Inspect retrieved nodes first, then chunking, embeddings, K and synthesis.

</details>

## Classic Architecture Q&A — Memorize This

### Q. Why do you use LangChain selectively instead of making it the entire application framework?

> **I use LangChain selectively rather than making it the entire application framework. LangChain's ChatOpenAI integration handles structured extraction, LangGraph handles stateful workflow orchestration, and LlamaIndex with FAISS handles the knowledge RAG layer. This keeps responsibilities explicit and prevents the LLM from directly controlling analytics SQL.**

<div style="background:#eef7ff;border:1px solid #c9e0f2;border-radius:10px;padding:10px 12px;margin:10px 0;">
</div>

### Memory Map

```text
LangChain / ChatOpenAI  → Structured Extraction
Pydantic                → Typed Contract
LangGraph               → Stateful Workflow
LlamaIndex + FAISS      → Knowledge RAG
Controlled SQL          → Trusted Numbers
FastAPI                 → Service API
```

### One-line takeaway

> **Do not force every responsibility into one framework. Keep the boundaries explicit.**